[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Daniel-534/IntroduccionAstronomiaPractica/blob/main/CirculosPrincipales-CoordenadasCelestes/Analisis.ipynb)

In [1]:
!pip install astroquery -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.2 MB/s eta 0:00:00


In [16]:
"""
Sun Ephemeris Query — JPL Horizons via astroquery
==================================================
Target  : Sol (Sun) [ID=10]
Observer: 6.2675°N, 75.2675°W, 1495 m  (lon/lat/elev)
Period  : 2026-04-03 00:00 UT -> 2026-04-04 00:00 UT
Step    : 1 minute  (1441 epochs)

Columns retrieved
-----------------
  datetime_str   — epoch label from Horizons
  datetime_jd    — Julian Date
  RA             — Astrometric Right Ascension  [deg, J2000/ICRF]
  DEC            — Astrometric Declination       [deg, J2000/ICRF]
  AZ             — Apparent Azimuth              [deg, N->E convention]
  EL             — Apparent Elevation            [deg]

Quantities used (Horizons codes)
---------------------------------
  1  -> Astrometric RA & DEC (J2000/ICRF)
  4  -> Apparent AZ & EL (airless, i.e. no refraction correction)

Install requirements
--------------------
  pip install astroquery pandas
"""

import pandas as pd
from astroquery.jplhorizons import Horizons

# ── Observer location ────────────────────────────────────────────────────────
# Format expected by Horizons: {'lon': deg_E, 'lat': deg_N, 'elevation': km}
# Note: negative latitude → Southern Hemisphere
LOCATION = {
    "lon": -75.267452683656837,   # degrees East
    "lat": 6.267452683656837,  # degrees North (negative = South)
    "elevation": 1.495,               # km above WGS-84 ellipsoid
}

# ── Epochs ───────────────────────────────────────────────────────────────────
EPOCHS = {
    "start": "2026-04-03 00:00",
    "stop":  "2026-04-04 00:00",
    "step":  "10m",               # 1-minute cadence
}

# ── Query ────────────────────────────────────────────────────────────────────
print("Querying JPL Horizons …")
obj = Horizons(
    id="10",           # Sun
    location=LOCATION,
    epochs=EPOCHS,
)

eph = obj.ephemerides(
    quantities="1,4",
    skip_daylight=False,
)

# ── Build DataFrame ──────────────────────────────────────────────────────────
# Convert AstroPy table -> pandas, then keep only the columns we care about
df_raw = eph.to_pandas()

df = df_raw[["datetime_str", "datetime_jd", "RA", "DEC", "AZ", "EL"]].copy()
df["Epoch (UT)"] = pd.to_datetime(df_raw["datetime_str"], format="%Y-%b-%d %H:%M")
df["Epoch (COL)"] = df["Epoch (UT)"].dt.tz_localize("UTC").dt.tz_convert("America/Bogota")

# ── Inspect & save ───────────────────────────────────────────────────────────
print(f"\nShape  : {df.shape[0]} rows × {df.shape[1]} columns")
display(df)

Querying JPL Horizons …

Shape  : 145 rows × 8 columns


,datetime_str,datetime_jd,RA,DEC,AZ,EL,Epoch (UT),Epoch (COL)
0,2026-Apr-03 00:00,2.461134e+06,11.85194,5.08784,276.883137,-13.158144,2026-04-03 00:00:00,2026-04-02 19:00:00-05:00
1,2026-Apr-03 00:10,2.461134e+06,11.85830,5.09049,277.237273,-15.624481,2026-04-03 00:10:00,2026-04-02 19:10:00-05:00
2,2026-Apr-03 00:20,2.461134e+06,11.86466,5.09315,277.610359,-18.088811,2026-04-03 00:20:00,2026-04-02 19:20:00-05:00
3,2026-Apr-03 00:30,2.461134e+06,11.87103,5.09580,278.004549,-20.550917,2026-04-03 00:30:00,2026-04-02 19:30:00-05:00
4,2026-Apr-03 00:40,2.461134e+06,11.87740,5.09846,278.422298,-23.010548,2026-04-03 00:40:00,2026-04-02 19:40:00-05:00
...,...,...,...,...,...,...,...,...
140,2026-Apr-03 23:20,2.461134e+06,12.73841,5.46015,276.010043,-3.304666,2026-04-03 23:20:00,2026-04-03 18:20:00-05:00
141,2026-Apr-03 23:30,2.461134e+06,12.74475,5.46280,276.306872,-5.775568,2026-04-03 23:30:00,2026-04-03 18:30:00-05:00
142,2026-Apr-03 23:40,2.461134e+06,12.75110,5.46544,276.616936,-8.245008,2026-04-03 23:40:00,2026-04-03 18:40:00-05:00
143,2026-Apr-03 23:50,2.461134e+06,12.75745,5.46808,276.941608,-10.712843,2026-04-03 23:50:00,2026-04-03 18:50:00-05:00


In [17]:
df_filtrado = df.set_index("Epoch (COL)").between_time("12:00", "15:00").reset_index()
df_filtrado

,Epoch (COL),datetime_str,datetime_jd,RA,DEC,AZ,EL,Epoch (UT)
0,2026-04-03 12:00:00-05:00,2026-Apr-03 17:00,2.461134e+06,12.50020,5.35950,125.880810,88.694153,2026-04-03 17:00:00
1,2026-04-03 12:10:00-05:00,2026-Apr-03 17:10,2.461134e+06,12.50642,5.36216,241.971517,88.378731,2026-04-03 17:10:00
2,2026-04-03 12:20:00-05:00,2026-Apr-03 17:20,2.461134e+06,12.51265,5.36481,259.212213,86.009244,2026-04-03 17:20:00
3,2026-04-03 12:30:00-05:00,2026-Apr-03 17:30,2.461134e+06,12.51888,5.36747,263.584648,83.550375,2026-04-03 17:30:00
4,2026-04-03 12:40:00-05:00,2026-Apr-03 17:40,2.461134e+06,12.52511,5.37012,265.611462,81.075742,2026-04-03 17:40:00
5,2026-04-03 12:50:00-05:00,2026-Apr-03 17:50,2.461134e+06,12.53134,5.37278,266.815244,78.595643,2026-04-03 17:50:00
6,2026-04-03 13:00:00-05:00,2026-Apr-03 18:00,2.461134e+06,12.53756,5.37543,267.636632,76.113050,2026-04-03 18:00:00
7,2026-04-03 13:10:00-05:00,2026-Apr-03 18:10,2.461134e+06,12.54380,5.37809,268.249822,73.629144,2026-04-03 18:10:00
8,2026-04-03 13:20:00-05:00,2026-Apr-03 18:20,2.461134e+06,12.55003,5.38074,268.737456,71.144490,2026-04-03 18:20:00
9,2026-04-03 13:30:00-05:00,2026-Apr-03 18:30,2.461134e+06,12.55626,5.38339,269.143805,68.659398,2026-04-03 18:30:00
